# Primality Testing
The purpose of this notebook is to use different methods from machine learning to create primality testers. This is mainly for fun / gimicks.

# Project 1
Make a neural network that takes in a binary string and correctly outputs the primality of the input.
### Goals:
- The resulting network should return a correct answer for every single binary string input.
- The resulting network should have as few parameters as possible.
- The resulting network should take as input as large of a binary string as possible.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on {device}")

torch.manual_seed(0)
loss_fn = torch.nn.BCEWithLogitsLoss()

In [ ]:
# Build the datasets and dataloaders

In [ ]:
# Accuracy and validation functions
def validation(net, val_loader):
    net.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device)

            y_hat = net(x)
            loss = loss_fn(y_hat, y)

            total_loss += loss.item() * x.size(0)
            total_correct += ((torch.sigmoid(y_hat) > 0.5) == y.squeeze(-1)).sum().item()
            total_samples += x.size(0)

    avg_loss = total_loss / total_samples
    avg_accuracy = total_correct / total_samples

    # Putting our model back into train mode
    net.train()
    return avg_loss, avg_accuracy

def get_accuracy(y_hat, y):
    # We use tensor broadcasting here
    return torch.mean(((torch.sigmoid(y_hat) > 0.5) == y.squeeze(-1)).float())

In [ ]:
# Making the network class
class MLP(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
def train(net, train_loader, val_loader, optimizer, n_optimization_steps, log_val_interval):
    net = net.to(device)
    losses_train = []
    losses_valid = []
    accuracies_train = []
    accuracies_valid = []
    index = 0

    while True:
        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)
            y_hat = net(x)

            # Compute the loss function
            L = loss_fn(y_hat, y)
            losses_train.append(L.item())
            accuracies_train.append(get_accuracy(y_hat, y).item())

            # Compute the L gradient and using the optimizer
            L.backward()
            optimizer.step()
            optimizer.zero_grad()

            index += 1
            if index % log_val_interval == 0:
                loss, accuracy = validation(net, val_loader)
                losses_valid.append(loss)
                accuracies_valid.append(accuracy)

            if index >= n_optimization_steps:
                return losses_train, accuracies_train, losses_valid, accuracies_valid

In [ ]:
# Training the MLP
